# 🏡 Min-Max Normalization Workshop
## Team Name/Number: [Team Number]
## Team Members: [Full Name 1], [Full Name 2], [Full Name 3]
---

## ❗ Why We Normalize: The Problem with Raw Feature Scales

In housing data, features like `Price` and `Lot_Size` can have values in the hundreds of thousands, while others like `Num_Bedrooms` range from 1 to 5. This creates problems when we use algorithms that depend on numeric magnitudes.

---

### ⚠️ What Goes Wrong Without Normalization

---

### 1. 🧭 K-Nearest Neighbors (KNN)

KNN uses the **Euclidean distance** formula:

$$
d = \sqrt{(x_1 - x_2)^2 + (y_1 - y_2)^2 + \cdots}
$$

**Example:**

- $ \text{Price}_1 = 650{,}000, \quad \text{Price}_2 = 250{,}000 $
- $ \text{Bedrooms}_1 = 3, \quad \text{Bedrooms}_2 = 2 $

Now compute squared differences:

$$
(\text{Price}_1 - \text{Price}_2)^2 = (650{,}000 - 250{,}000)^2 = (400{,}000)^2 = 1.6 \times 10^{11}
$$
$$
(\text{Bedrooms}_1 - \text{Bedrooms}_2)^2 = (3 - 2)^2 = 1
$$

➡️ **Price dominates the distance calculation**, making smaller features like `Bedrooms` irrelevant.

---

### 2. 📉 Linear Regression

Linear regression estimates:

$$
y = \beta_1 \cdot \text{Price} + \beta_2 \cdot \text{Bedrooms} + \beta_3 \cdot \text{Lot\_Size} + \epsilon
$$

If `Price` has very large values:
- Gradient updates for $ \beta_1 $ will be **much larger**
- Gradient updates for $ \beta_2 $ (Bedrooms) will be **very small**

➡️ The model overfits high-magnitude features like `Price`.

---

### 3. 🧠 Neural Networks

A single neuron computes:

$$
z = w_1 \cdot \text{Price} + w_2 \cdot \text{Bedrooms} + w_3 \cdot \text{Lot\_Size}
$$

If:

- $ \text{Price} = 650{,}000 $
- $ \text{Bedrooms} = 3 $
- $ \text{Lot\_Size} = 8{,}000 $

Then:

$$
z \approx w_1 \cdot 650{,}000 + w_2 \cdot 3 + w_3 \cdot 8{,}000
$$

➡️ Even with equal weights, `Price` contributes **most of the activation**, making it difficult for the network to learn from other features.

---

### ✅ Solution: Min-Max Normalization

We apply the transformation:

$$
x_{\text{normalized}} = \frac{x - x_{\text{min}}}{x_{\text{max}} - x_{\text{min}}}
$$

This scales all features to a common range (typically $[0, 1]$).

| Feature      | Raw Value | Min     | Max     | Normalized Value |
|--------------|-----------|---------|---------|------------------|
| Price        | 650,000   | 250,000 | 800,000 | 0.72             |
| Bedrooms     | 3         | 1       | 5       | 0.50             |
| Lot_Size     | 8,000     | 3,000   | 10,000  | 0.714            |

➡️ Now, **each feature contributes fairly** to model training or distance comparisons.

---

## 📌 Use Case: Housing Data
We are normalizing features from a real estate dataset to prepare it for machine learning analysis.

In [1]:
# 🔢 Load and display dataset
import pandas as pd
df = pd.read_csv('./data/housing_data.csv')
df.head()

,House_ID,Price,Area_sqft,Num_Bedrooms,Num_Bathrooms,Year_Built,Lot_Size
0,H100000,574507,1462,3,3,2002,4878
1,H100001,479260,1727,2,2,1979,4943
2,H100002,597153,1403,5,2,1952,5595
3,H100003,728454,1646,5,2,1992,9305
4,H100004,464876,853,1,1,1956,7407


### 🔎 Step 1 — Implement Min-Max Normalization on the Housing Dataset

In [2]:
# ✍️ Implement Min-Max Normalization manually here (no sklearn/numpy)
# Normalize: Price, Area_sqft, Num_Bedrooms, Num_Bathrooms, Lot_Size

def min_max_normalize(values):
    """Scale a list of numbers to the range [0, 1] using (x - min) / (max - min)."""
    col_min = min(values)
    col_max = max(values)
    value_range = col_max - col_min

    # Edge case: constant column (max == min) -> avoid division by zero
    if value_range == 0:
        return [0.0 for _ in values]

    return [(x - col_min) / value_range for x in values]


features_to_normalize = ['Price', 'Area_sqft', 'Num_Bedrooms', 'Num_Bathrooms', 'Lot_Size']

for col in features_to_normalize:
    df[col + '_norm'] = min_max_normalize(df[col].tolist())

# Check: every normalized column should have min 0 and max 1
norm_cols = [col + '_norm' for col in features_to_normalize]
print(df[norm_cols].agg(['min', 'max']).round(3))
print()

# Before vs. after: compare the ranges
for col in features_to_normalize:
    print(f"{col:15} raw range: {df[col].max() - df[col].min():>10,}   normalized range: {df[col + '_norm'].max() - df[col + '_norm'].min():.1f}")

df[['House_ID'] + norm_cols].head()

     Price_norm  Area_sqft_norm  Num_Bedrooms_norm  Num_Bathrooms_norm  \
min         0.0             0.0                0.0                 0.0   
max         1.0             1.0                1.0                 1.0   

     Lot_Size_norm  
min            0.0  
max            1.0  

Price           raw range:    977,909   normalized range: 1.0
Area_sqft       raw range:      3,363   normalized range: 1.0
Num_Bedrooms    raw range:          4   normalized range: 1.0
Num_Bathrooms   raw range:          2   normalized range: 1.0
Lot_Size        raw range:     12,088   normalized range: 1.0


,House_ID,Price_norm,Area_sqft_norm,Num_Bedrooms_norm,Num_Bathrooms_norm,Lot_Size_norm
0,H100000,0.485226,0.315789,0.50,1.0,0.320814
1,H100001,0.387827,0.394588,0.25,0.5,0.326191
2,H100002,0.508384,0.298246,1.00,0.5,0.380129
3,H100003,0.642651,0.370503,1.00,0.5,0.687045
4,H100004,0.373119,0.134701,0.00,0.0,0.530030


### 🧪 Evidence: Feature Contributions to KNN Distance (Before vs. After)

To see why normalization matters for KNN, we compute the Euclidean distance between the first two houses and measure what share of that distance each feature contributes.

In [3]:
# Share of the squared Euclidean distance between house 0 and house 1 contributed by each feature
def distance_shares(row_a, row_b, columns):
    squared_diffs = {col: (row_a[col] - row_b[col]) ** 2 for col in columns}
    total = sum(squared_diffs.values())
    return {col: 100 * sq / total for col, sq in squared_diffs.items()}

house_a, house_b = df.iloc[0], df.iloc[1]
raw_shares = distance_shares(house_a, house_b, features_to_normalize)
norm_shares = distance_shares(house_a, house_b, norm_cols)

comparison = pd.DataFrame({
    'Raw share of distance (%)': [raw_shares[col] for col in features_to_normalize],
    'Normalized share of distance (%)': [norm_shares[col + '_norm'] for col in features_to_normalize],
}, index=features_to_normalize)
comparison.round(4)

,Raw share of distance (%),Normalized share of distance (%)
Price,99.9992,2.8902
Area_sqft,0.0008,1.8918
Num_Bedrooms,0.0000,19.0418
Num_Bathrooms,0.0000,76.1673
Lot_Size,0.0000,0.0088


### 🔎 Talking Points #1 — Why Normalization Matters for Our Housing Data

- **KNN: without normalization, distance just measures price.** Between the first two houses in our dataset, `Price` accounts for **99.999%** of the squared Euclidean distance, while bedrooms, bathrooms and lot size together contribute almost nothing. After Min-Max normalization the shares are spread out (Bathrooms 76%, Bedrooms 19%, Price 3%, Area 2%), so every feature has a real say in which houses count as "neighbors."

- **Gradient-based learning: equal scales mean balanced, stable updates.** In gradient descent, the gradient for each weight is multiplied by that feature's value. Raw `Price` values (100,000–1,077,909) produce gradients thousands of times larger than `Num_Bathrooms` (1–3), so one learning rate is either too large for Price (training diverges) or too small for Bathrooms (it barely learns). With every feature in [0, 1], one learning rate works for all weights and training converges faster.

- **Housing features use very different units, and normalization removes that bias.** Before normalization the ranges were Price ≈ 977,909, Lot_Size ≈ 12,088, Area_sqft ≈ 3,363, Bedrooms = 4 and Bathrooms = 2. After normalization every feature has a range of exactly 1. Our implementation also handles the edge case of a constant column (max = min) by returning 0 instead of dividing by zero, and it keeps the original columns (adding `_norm` versions) so the raw values remain available for interpretation.


## 🧩 Challenge Extension: After Normalization, Which Features Matter Most?

You’ve normalized the housing features so they share a common scale.  
Now comes a common next step in ML workflows:

> **How do we identify the most important directions (principal components) in the data—and how might these relate to a target variable like `Price`?**

This introduces **Principal Component Analysis (PCA)**.

---

## 📚 PCA Theory (Conceptual)

### What PCA *is*
PCA is an **unsupervised** dimensionality reduction technique that:
- Finds **new axes** (principal components) that are **linear combinations** of your original features.
- Orders these axes so that:
  - **PC1** captures the **most variance** in the feature space,
  - **PC2** captures the next most variance, and so on,
  - Each PC is **orthogonal** (uncorrelated) with the previous ones.

### What PCA is *not*
PCA does **not** directly find features that “impact the target variable” because it does not use the target in its optimization.

However, you *can*:
- Compute PCs from the feature matrix **X** (after normalization),
- Then measure how PCs relate to the target **y** (e.g., correlation with `Price`, or a simple regression on PCs),
- Interpret which original features contribute most to PCs that are most related to **y**.

---

## 🧠 The Math (high level)
Given a centered feature matrix \(X\) (often standardized/normalized first):

1. Compute covariance matrix:
$$
\Sigma = \frac{1}{n-1}X^\top X
$$

2. Find eigenvectors (principal directions) and eigenvalues (variance captured):
$$
\Sigma v_i = \lambda_i v_i
$$

- $ v_i $ are **principal component directions** (loadings)
- $ \lambda_i $ are the **variance explained** by each component

---

## ✅ Why Normalize Before PCA?
PCA is sensitive to scale. Without normalization/standardization:
- A large-scale feature (e.g., `Price`) can dominate variance
- PCs will reflect units rather than structure

---

## 🎯 Student Challenge
Using the **housing dataset**:

1. Apply PCA to the normalized feature matrix \(X\) (exclude ID columns and the target).
2. Determine how many components are needed to explain **≥ 90%** of the variance.
3. Identify which original features contribute most to:
   - **PC1** and **PC2**, and
   - the **PC most correlated with the target** (`Price`).
4. Write a short interpretation:
   - “What does PC1 represent in housing terms?”
   - “Do the PCs that explain the most variance also relate most strongly to `Price`?”



### 🔗 How to Integrate This With Your Step 1 Normalization

- If you created normalized columns (e.g., `Area_sqft_norm`), use those in `candidate_features`.
- If you normalized in-place (overwriting original columns), you can use the original names.
- PCA should **not** include:
  - `House_ID` (identifier)
  - non-numeric categorical columns (unless encoded appropriately)
- Decide intentionally whether to include `Year_Built`:
  - It’s numeric, but it may behave differently than size/price-related features.

---

### ✅ Deliverable for the Challenge
Add a Markdown cell answering:

1. How many PCs explain at least **90%** variance?
2. Which features contribute most to **PC1** and **PC2**?
3. Which PC is most correlated with `Price`?
4. In plain language: what do you think PC1 represents?


In [4]:
# --- PCA Scaffolding (after you have a normalized DataFrame) ---
import pandas as pd
import numpy as np

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# ---- Step A: Choose target and feature columns ----
# Price is the target, so it is excluded from X. House_ID is an identifier, so it is excluded too.
# Year_Built is excluded: it is an age/time feature, not a size feature, and was not part of Step 1.
target = 'Price'
candidate_features = ['Area_sqft_norm', 'Num_Bedrooms_norm', 'Num_Bathrooms_norm', 'Lot_Size_norm']

X = df[candidate_features]
y = df[target]

# ---- Step B: Ensure scaling appropriate for PCA ----
# Features are already Min-Max normalized to [0, 1], so no extra scaling is applied.
# PCA centers each column (subtracts its mean) automatically.
X_scaled = X.values

# ---- Step C: Fit PCA ----
pca = PCA()
X_pca = pca.fit_transform(X_scaled)
pc_names = [f'PC{i + 1}' for i in range(pca.n_components_)]

# ---- Step D: Variance explained ----
variance_df = pd.DataFrame({
    'Explained Variance': pca.explained_variance_ratio_,
    'Cumulative Variance': np.cumsum(pca.explained_variance_ratio_)
}, index=pc_names)
print(variance_df.round(3))

n_components_90 = int(np.argmax(variance_df['Cumulative Variance'].values >= 0.90) + 1)
print(f"\nComponents needed to explain >= 90% variance: {n_components_90}")

# ---- Step E: Loadings (feature contributions to PCs) ----
loadings = pd.DataFrame(pca.components_.T, index=candidate_features, columns=pc_names)
print("\nLoadings:")
print(loadings.round(3))

for pc in ['PC1', 'PC2']:
    top = loadings[pc].abs().sort_values(ascending=False)
    print(f"\nTop contributors to {pc}:")
    print(top.round(3))

# ---- Step F: Relate PCs to the target (simple correlation) ----
pc_scores = pd.DataFrame(X_pca, columns=pc_names)
pc_target_corr = pc_scores.corrwith(y)
print("\nCorrelation of each PC with Price:")
print(pc_target_corr.round(3))

best_pc = pc_target_corr.abs().idxmax()
print(f"\nPC most correlated with Price: {best_pc} (r = {pc_target_corr[best_pc]:.3f})")
print(f"Top contributors to {best_pc}:")
print(loadings[best_pc].abs().sort_values(ascending=False).round(3))

     Explained Variance  Cumulative Variance
PC1               0.495                0.495
PC2               0.360                0.855
PC3               0.080                0.936
PC4               0.064                1.000

Components needed to explain >= 90% variance: 3

Loadings:
                      PC1    PC2    PC3    PC4
Area_sqft_norm      0.002  0.017 -0.124  0.992
Num_Bedrooms_norm   0.053  0.998  0.020 -0.015
Num_Bathrooms_norm  0.999 -0.053 -0.005 -0.001
Lot_Size_norm       0.004 -0.019  0.992  0.124

Top contributors to PC1:
Num_Bathrooms_norm    0.999
Num_Bedrooms_norm     0.053
Lot_Size_norm         0.004
Area_sqft_norm        0.002
Name: PC1, dtype: float64

Top contributors to PC2:
Num_Bedrooms_norm     0.998
Num_Bathrooms_norm    0.053
Lot_Size_norm         0.019
Area_sqft_norm        0.017
Name: PC2, dtype: float64

Correlation of each PC with Price:
PC1   -0.011
PC2   -0.005
PC3    0.017
PC4   -0.015
dtype: float64

PC most correlated with Price: PC3 (r = 0.017)
T

### 🧪 Evidence: PCA on Raw vs. Normalized Features

To show why scaling matters for PCA, we fit PCA on the same four features **before** normalization and compare the explained variance.

In [5]:
raw_features = ['Area_sqft', 'Num_Bedrooms', 'Num_Bathrooms', 'Lot_Size']
pca_raw = PCA().fit(df[raw_features])

pd.DataFrame({
    'Raw: Explained Variance': pca_raw.explained_variance_ratio_,
    'Normalized: Explained Variance': pca.explained_variance_ratio_,
}, index=pc_names).round(3)

,Raw: Explained Variance,Normalized: Explained Variance
PC1,0.941,0.495
PC2,0.059,0.360
PC3,0.000,0.080
PC4,0.000,0.064


In [6]:
# Largest loading on raw PC1 -> which feature dominates when data is not scaled
raw_pc1 = pd.Series(pca_raw.components_[0], index=raw_features)
print("Raw PC1 loadings:")
print(raw_pc1.abs().sort_values(ascending=False).round(3))

Raw PC1 loadings:
Lot_Size         1.000
Area_sqft        0.008
Num_Bedrooms     0.000
Num_Bathrooms    0.000
dtype: float64


### ✅ Challenge Deliverable — Answers

1. **How many PCs explain at least 90% variance?**
   **3 components.** PC1 explains 49.5%, PC2 36.0% and PC3 8.0%, for a cumulative 93.6%. (PC4 adds the remaining 6.4%.)

2. **Which features contribute most to PC1 and PC2?**
   - **PC1:** `Num_Bathrooms` (loading ≈ 0.999). All other features are near 0.
   - **PC2:** `Num_Bedrooms` (loading ≈ 0.998). All other features are near 0.

3. **Which PC is most correlated with `Price`?**
   **PC3** (driven almost entirely by `Lot_Size`, loading ≈ 0.992), but the correlation is only **r ≈ 0.017**, which is essentially zero. No PC has a meaningful relationship with Price (all |r| < 0.02).

4. **In plain language, what does PC1 represent?**
   PC1 represents **the number of bathrooms in the house**. It is not a combined "house size" factor, because the features in this dataset are uncorrelated with each other, so PCA cannot find shared patterns to combine. Bathrooms land in PC1 because they have only 3 values (0, 0.5, 1 after normalizing), which spreads them evenly across [0, 1] and gives them the largest variance of any feature.

**Do the PCs that explain the most variance also relate most strongly to `Price`?**
No. PC1 and PC2 explain 85.5% of the variance but have almost no correlation with Price. PCA is unsupervised, so it finds directions of spread in the features without considering the target.

### 🔎 Talking Points #2 — What PCA Revealed About the Housing Features

- **Normalization changed the PCA results completely.** On the raw features, PC1 explained about 94% of the variance only because `Lot_Size` has the largest numbers (range ≈ 12,000). It reflected the units, not real structure in the data. After Min-Max normalization, every feature has a range of 1, and the variance is spread across components (PC1 49.5%, PC2 36.0%, PC3 8.0%, PC4 6.4%). We needed **3 of 4 components** to reach ≥ 90% variance, so PCA could not compress the data much.

- **Each principal component is basically one original feature.** PC1 is almost entirely `Num_Bathrooms` (loading 0.999), PC2 is `Num_Bedrooms` (0.998), PC3 is `Lot_Size` (0.992) and PC4 is `Area_sqft` (0.992). This happens when features are uncorrelated with each other, so there are no shared patterns for PCA to combine. Bathrooms rank first because they take only 3 values (0, 0.5, 1 after normalizing), which spreads them evenly across the range and gives them the highest variance. In housing terms, PC1 is "number of bathrooms," not an overall "house size" factor.

- **The components that explain the most variance are not the ones that relate to Price.** PCA is unsupervised: it never sees the target. The PC most correlated with `Price` was PC3 (Lot_Size), but with r = 0.017, which is essentially zero, and every PC had |r| < 0.02. In this dataset, high variance does not mean high predictive importance. To find the features that drive Price, we would need a supervised method (e.g. regression coefficients or feature importance) instead of PCA alone.

# 📤 How to Submit the Assignment

Follow these instructions carefully to ensure your team submits a complete and well-organized assignment.

## Step 1: Get the Starter Repository

1. Clone the GitHub repository provided by the instructor.
2. Keep the **original repository name**. Do **not** rename the repository.

## Step 2: Complete the Workshop

3. Create **one shared Jupyter Notebook** for the entire team to work on.
4. At the top of **`MinMax_Normalization_Workshop.ipynb`**, clearly include:
   - Team Number
   - Full Name of each team member
5. Complete **all coding tasks, analyses, and written answers** in the same notebook.
6. Make sure all notebook cells execute successfully before submission.

## Step 3: Update the Project Documentation

7. Replace the existing **`README.md`** file with a new version that:
   - Introduces your team
   - Explains the purpose of the workshop
   - Describes the work you completed
   - Summarizes your findings and conclusions
   - Can be used as the visual aid for your team's **5-minute presentation**

## Step 4: Review Project Dependencies

8. Review the project's Python dependencies and decide whether the **`requirements.txt`** file needs to be updated.
   - If you installed additional packages, regenerate or update the file.
   - If no new dependencies were added, no changes may be necessary.

## Step 5: Publish Your Work

9. Push the completed repository to your own GitHub account.
10. Verify that all files have been uploaded correctly and that the repository is publicly accessible if required by the instructor.

---

## ✅ Submission Checklist

Before submitting, confirm that the following items are complete:

- [ ] Repository name has not been changed
- [ ] `MinMax_Normalization_Workshop.ipynb` contains all required work
- [ ] Team number and team member names appear at the top of the notebook
- [ ] All questions have been answered
- [ ] All code runs successfully
- [ ] `README.md` has been replaced with a project summary
- [ ] README can support a 5-minute presentation
- [ ] `requirements.txt` has been reviewed and updated if necessary
- [ ] Repository has been pushed to your GitHub account
- [ ] Final work has been reviewed by all team members

> **Important:** Submit a repository that is clean, complete, and easy to understand. The notebook and README should clearly demonstrate your team's work, findings, and understanding of Min-Max Normalization.